# Encontro 6. População, amostra e o sorteio

**Disciplina:** Métodos e Técnicas de Pesquisa Quantitativa — Administração/UFMA
**Docente:** Prof. Dr. Tadeu Gomes Teixeira

> **O que você vai fazer hoje**
> 1. **Executar** as células de cima para baixo.
> 2. **Mudar um valor** no bloco de configuração: o tamanho da amostra.
> 3. **Escrever** a diferença entre a população e o cadastro, no fim.
>
> Você não vai escrever código. Se algo não funcionar, **chame o professor**.

---

## Painel do encontro

Hoje a pergunta é **“de quem esses dados falam?”**. E, pela primeira vez, a resposta
depende de uma decisão sua: **quais casos entram na amostra, e como eles foram escolhidos.**

A base de hoje tem uma particularidade: ela é, ao mesmo tempo, **a população** e **o
cadastro** do exercício. É a lista de postos de combustível do Maranhão que a ANP
pesquisa todo mês — cada posto com CNPJ, município e bandeira.

| Item | Valor |
|---|---|
| **Fonte** | ANP — Série Histórica de Preços de Combustíveis (SHPC) |
| **O que é** | O **cadastro** de postos: um registro por posto, com CNPJ e município |
| **Recorte** | Maranhão, 2025, gasolina comum |
| **População do exercício** | **236 postos**, em 11 municípios |
| **Unidade** | R$ por litro (o preço médio do posto no ano) |

> **O que a disciplina faz com esta base hoje** é o que se faz com um cadastro de
> empresas: **sortear**. E o que se aprende aqui vale para qualquer cadastro — o CEMPRE,
> a RAIS, a base de clientes de uma empresa.

## Bloco de configuração

**O único lugar do notebook onde se mexe em código.**

```python
# --- o que você pode mudar ---
N_AMOSTRA = 100   # quantos postos sortear, de 10 a 200
```

In [1]:
N_AMOSTRA = 100   # quantos postos sortear, de 10 a 200

print("Tamanho da amostra de hoje:", N_AMOSTRA)

Tamanho da amostra de hoje: 100


## Utilitários da disciplina

`conferir` compara o seu resultado com o número do slide. **Não precisa alterá-la.**

In [2]:

# =====================================================================
#  UTILITARIOS DA DISCIPLINA - nao precisa alterar nada daqui para baixo
#  estas duas funcoes sao usadas pelas celulas do encontro. Se voce nao
#  sabe o que elas fazem, tudo bem: nao mexa. Voce so precisa executar.
# =====================================================================

def conferir(descricao, obtido, esperado, tolerancia=0.005, unidade="%"):
    """Confere o resultado calculado contra o valor ja resolvido no slide.

    Chame assim:  conferir("taxa de sobrevivencia do setor X", obtido, 0.412)

    O 'esperado' e o numero que ja estava escrito no slide, resolvido a mao
    pelo professor. Se aparecer [ok], o seu calculo esta certo. Se aparecer
    [X], confira a base, o filtro e a coluna que voce usou.
    """
    try:
        dif = abs(float(obtido) - float(esperado))
    except (TypeError, ValueError):
        print("  [X ] nao deu para comparar: obtido = %r" % (obtido,))
        return
    num = lambda v: ("%.*f%s" % (2, v, unidade)) if unidade else ("%.4f" % v)
    if dif <= tolerancia:
        print("  [ok] %-54s obtido %-9s esperado %s"
              % (descricao[:54], num(float(obtido)), num(float(esperado))))
    else:
        print("  [X ] %-54s obtido %-9s esperado %-9s dif %s"
              % (descricao[:54], num(float(obtido)), num(float(esperado)), num(dif)))


def cartao(df, coluna, nome_base="base"):
    """Imprime o contexto que voce precisa colar no assistente de IA.

    Chame assim:  cartao(dados, "porte")

    Copie a saida inteira e cole no pedido. E isso que impede a IA de
    inventar numero: ela so pode usar o que esta escrito no cartao.
    """
    print("=== CARTAO DE CONTEXTO (copie tudo daqui para o assistente) ===")
    print("base: %s" % nome_base)
    print("linhas x colunas: %d x %d" % df.shape)
    print("colunas e seu significado:")
    for c in df.columns:
        print("   - %s (%s)" % (c, df[c].dtype))
    if coluna in df.columns:
        print("valores unicos de '%s': %s"
              % (coluna, sorted(map(str, df[coluna].dropna().unique()))[:25]))
    print("=== fim do cartao ===")


print("Utilitarios carregados: conferir() e cartao() estao prontas para uso.")


Utilitarios carregados: conferir() e cartao() estao prontas para uso.


## Seção 1. De quem esses dados falam

Antes de sortear, três definições que a disciplina vai usar o semestre inteiro:

| Termo | O que é | No exercício de hoje |
|---|---|---|
| **População** | O conjunto sobre o qual se quer falar | Os 236 postos do Maranhão |
| **Cadastro** | A lista a que se tem acesso, de onde se sorteia | A mesma lista dos 236 postos |
| **Amostra** | O subconjunto efetivamente estudado | Os postos sorteados |
| **Parâmetro** | A medida da população &mdash; o valor verdadeiro, quase sempre desconhecido | O preço médio dos 236 postos |
| **Estatística** | A medida da amostra &mdash; a estimativa | O preço médio dos postos sorteados |

> **População-alvo ≠ cadastro**, e é aí que nasce o primeiro viés. Aqui os dois
> coincidem de propósito, para que se veja o sorteio funcionando. Em um cadastro de
> empresas, eles quase nunca coincidem: o cadastro inclui só as formais, só as que
> declararam, e às vezes só as que estão ativas.

In [3]:
# ================= CÉLULA A: o cadastro, montado a partir dos dados da ANP =====
# Se esta célula falhar, peça ao professor para usar a CÉLULA B.

import io
import urllib.request

import pandas as pd

BASE_ANP = ("https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/"
            "arquivos/shpc/dsan/2025/precos-gasolina-etanol-%02d.csv")


def baixa_mes(mes):
    pedido = urllib.request.Request(BASE_ANP % mes,
                                    headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(pedido, timeout=180) as resposta:
        bruto = resposta.read()
    d = pd.read_csv(io.BytesIO(bruto), sep=";", encoding="utf-8-sig", decimal=",")
    d["mes"] = mes
    return d[d["Estado - Sigla"] == "MA"].copy()


quadros = [baixa_mes(m) for m in range(1, 13)]
anp = pd.concat(quadros, ignore_index=True)
anp = anp.rename(columns={"Municipio": "municipio", "CNPJ da Revenda": "cnpj",
                          "Produto": "produto", "Valor de Venda": "valor_venda",
                          "Bandeira": "bandeira"})
anp["valor_venda"] = pd.to_numeric(anp["valor_venda"], errors="coerce")

# O CADASTRO: um registro por posto, com o preço do primeiro mês em que aparece.
# A chave é o CNPJ — e não a combinação de município e bandeira, que muda de grafia
# entre os arquivos e criaria postos duplicados no cadastro.
gas = anp[anp["produto"] == "GASOLINA"].dropna(subset=["valor_venda"])
cadastro = (gas.sort_values("mes")
               .drop_duplicates("cnpj", keep="first")
               [["cnpj", "municipio", "bandeira", "valor_venda"]]
               .rename(columns={"valor_venda": "preco_medio"})
               .reset_index(drop=True))

N = len(cadastro)
print(f"CADASTRO montado: {N} postos, em {cadastro['municipio'].nunique()} municípios")
print(f"  preço médio da POPULAÇÃO: R$ {cadastro['preco_medio'].mean():.4f}")
print(f"  desvio padrão da POPULAÇÃO: R$ {cadastro['preco_medio'].std():.4f}")
cadastro.head(4)

CADASTRO montado: 236 postos, em 11 municípios
  preço médio da POPULAÇÃO: R$ 6.0087
  desvio padrão da POPULAÇÃO: R$ 0.3402


,cnpj,municipio,bandeira,preco_medio
0,02.962.193/0002-78,BARRA DO CORDA,BRANCA,6.25
1,14.662.197/0003-09,PINHEIRO,BRANCA,5.95
2,07.757.511/0003-63,CODO,BRANCA,6.34
3,17.600.625/0004-30,CODO,BRANCA,6.39


In [ ]:
# ================= CÉLULA B: CONTINGÊNCIA — leia antes =================
# NÃO EXECUTE AGORA. Só se o professor mandar, e depois de fazer upload do arquivo
# dados/anp_precos_revenda_ma_2025.csv.

import pandas as pd

anp = pd.read_csv("anp_precos_revenda_ma_2025.csv")
gas = anp[anp["produto"] == "GASOLINA"].dropna(subset=["valor_venda"])
cadastro = (gas.sort_values("mes_arquivo")
               .drop_duplicates("cnpj_da_revenda", keep="first")
               [["cnpj_da_revenda", "municipio", "bandeira", "valor_venda"]]
               .rename(columns={"cnpj_da_revenda": "cnpj",
                                "valor_venda": "preco_medio"})
               .reset_index(drop=True))
N = len(cadastro)
print(f"CADASTRO montado do arquivo local: {N} postos")
cadastro.head(4)

## Parte 2 · Estatística: o sorteio e a estimativa

**A ideia do bloco.** Uma amostra sorteada é uma **estimativa** de um valor que existe
na população, e essa estimativa **erra**. A pergunta do bloco é: quanto ela costuma
errar, e o que se pode fazer para errar menos.

**As quatro técnicas probabilísticas, em uma linha cada**

| Técnica | Como se faz | Quando usar |
|---|---|---|
| **Aleatória simples** | Sorteia n casos do cadastro inteiro | Cadastro completo e homogêneo |
| **Sistemática** | Sorteia o ponto de partida e toma 1 a cada k | Cadastro em ordem, sem periodicidade |
| **Estratificada** | Sorteia dentro de cada grupo, proporcional ao tamanho | Grupos internamente parecidos e diferentes entre si |
| **Por conglomerados** | Sorteia grupos inteiros | Cadastro disperso, coleta cara demais |

**A amostra tem erro, e ele pode ser medido.** O **erro de amostragem** é a diferença
entre a estatística (a média da amostra) e o parâmetro (a média da população). Ninguém
conhece o parâmetro na prática — e é por isso que este exercício é raro e valioso:
aqui ele é conhecido, e a turma pode **ver** o erro acontecer.

In [4]:
# A amostra aleatória simples: o sorteio honesto.
amostra = cadastro.sample(n=N_AMOSTRA, random_state=2025)

parametro = cadastro["preco_medio"].mean()      # o valor verdadeiro
estimativa = amostra["preco_medio"].mean()      # a estimativa da amostra
erro = estimativa - parametro

print(f"População : {N} postos | média (parâmetro) = R$ {parametro:.4f}")
print(f"Amostra   : {len(amostra)} postos | média (estimativa) = R$ {estimativa:.4f}")
print(f"Erro de amostragem = {erro:+.4f}")
print()
print("A estimativa NÃO é o valor verdadeiro: ela erra. E é isso que se mede.")

conferir("média da população", parametro, 6.0087, tolerancia=0.001, unidade=" R$")
conferir("média da amostra de 100", estimativa, 5.9753, tolerancia=0.001, unidade=" R$")

População : 236 postos | média (parâmetro) = R$ 6.0087
Amostra   : 100 postos | média (estimativa) = R$ 5.9753
Erro de amostragem = -0.0334

A estimativa NÃO é o valor verdadeiro: ela erra. E é isso que se mede.
  [ok] média da população                                     obtido 6.01 R$   esperado 6.01 R$
  [ok] média da amostra de 100                                obtido 5.98 R$   esperado 5.98 R$


### As três técnicas, lado a lado

Cada técnica constrói a amostra de um jeito, e cada uma dá uma estimativa diferente. O
exercício é comparar as três **com o mesmo valor verdadeiro ao lado**.

In [5]:
# As três probabilísticas, com o mesmo n, sobre o mesmo cadastro.
# 1. aleatória simples
aas = cadastro.sample(n=N_AMOSTRA, random_state=2025)

# 2. sistemática: um a cada k, depois do ponto de partida sorteado
k = N // N_AMOSTRA
partida = 7
passos = range(partida, N, k)
sist = cadastro.iloc[list(passos)][:N_AMOSTRA]

# 3. estratificada: proporcional ao tamanho de cada município
pesos = cadastro["municipio"].value_counts(normalize=True)
alocacao = (pesos * N_AMOSTRA).round().astype(int)
estrat = (cadastro.groupby("municipio", group_keys=False)
                 .apply(lambda g: g.sample(n=min(alocacao[g.name], len(g)),
                                           random_state=1)))

print(f"PARÂMETRO (a verdade): R$ {parametro:.4f}")
print()
print(f"  aleatória simples  n={len(aas):3d}  estimativa = R$ {aas['preco_medio'].mean():.4f}  "
      f"erro = {aas['preco_medio'].mean() - parametro:+.4f}")
print(f"  sistemática (k={k})  n={len(sist):3d}  estimativa = R$ {sist['preco_medio'].mean():.4f}  "
      f"erro = {sist['preco_medio'].mean() - parametro:+.4f}")
print(f"  estratificada      n={len(estrat):3d}  estimativa = R$ {estrat['preco_medio'].mean():.4f}  "
      f"erro = {estrat['preco_medio'].mean() - parametro:+.4f}")
print()
print("As três estimativas são diferentes, e as três erram um pouco.")
print("É isso que a margem de erro de uma pesquisa mede.")

PARÂMETRO (a verdade): R$ 6.0087

  aleatória simples  n=100  estimativa = R$ 5.9753  erro = -0.0334
  sistemática (k=2)  n=100  estimativa = R$ 6.0561  erro = +0.0474
  estratificada      n= 99  estimativa = R$ 6.0166  erro = +0.0078

As três estimativas são diferentes, e as três erram um pouco.
É isso que a margem de erro de uma pesquisa mede.


C:\Users\tadeu\AppData\Local\Temp\ipykernel_24764\3503222872.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(alocacao[g.name], len(g)),


In [6]:
# A amostra por conveniência: pegar os primeiros da lista, sem sortear.
conveniencia = cadastro.head(N_AMOSTRA)

print(f"  conveniência (os {N_AMOSTRA} primeiros)  estimativa = "
      f"R$ {conveniencia['preco_medio'].mean():.4f}  "
      f"erro = {conveniencia['preco_medio'].mean() - parametro:+.4f}")
print()
print("Repare no erro. Os primeiros da lista são os de um município, e o preço varia")
print("por município: o erro da conveniência é o maior dos quatro, e ele NÃO é aleatório.")
print("É um viés — e viés não se conserta aumentando a amostra.")

  conveniência (os 100 primeiros)  estimativa = R$ 6.1306  erro = +0.1219

Repare no erro. Os primeiros da lista são os de um município, e o preço varia
por município: o erro da conveniência é o maior dos quatro, e ele NÃO é aleatório.
É um viés — e viés não se conserta aumentando a amostra.


**A frase que vai para o relatório**

> "Sobre um cadastro de **236 postos** de combustível do Maranhão, cujo preço médio é
> **R$ 6,01**, uma amostra aleatória simples de 100 postos estimou o preço médio em
> **R$ 5,98** — um erro de amostragem de **−R$ 0,03**, ou 0,6% do valor verdadeiro
> (ANP, SHPC, 2025)."

**O que essa frase não diz.** Que a amostra "acertou". Ela errou, e o erro está declarado.
Uma amostra não é boa por acertar o valor verdadeiro — é boa por ter sido **sorteada**, e
por isso ter um erro que se pode **medir e declarar**.

## Seção 2. As três perguntas do dia

Responda por escrito, editando as células abaixo.

**1. Qual é a população-alvo do seu projeto, e qual é o cadastro a que você tem acesso?
A diferença entre os dois é grande?**

---


**2. Que técnica de amostragem caberia no seu projeto? Justifique pela forma do cadastro.**

---


**3. Se alguém usasse uma amostra por conveniência no seu projeto, que viés apareceria?**

---


## Antes de sair

- [ ] Executei o notebook inteiro de cima para baixo, sem erro
- [ ] Troquei `N_AMOSTRA` para outro valor e vi a estimativa mudar
- [ ] Comparei as quatro amostras e sei qual erra mais, e por quê
- [ ] Respondi as três perguntas, uma delas sobre o meu projeto

> Guarde este notebook. No próximo encontro ele responde a outra pergunta: se eu
> sorteasse **mil** amostras, como as estimativas se distribuíram?